In [ ]:
import numpy as np
import time
import os
from PIL import Image
import matplotlib.pyplot as plt
import scipy.stats

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Upload standard test images (Lena.png, Baboon.png) to your board first!
IMAGE_PATH = "Lena.png"  # Change this to your image filename
WEIGHTS_DIR = "."        # Directory where .npy weights are stored

# ESN Parameters
N_RESERVOIR = 1024
LEAK_RATE = 0.05
N_DROP = 1000  # Warmup steps to ignore

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def load_weights():
    try:
        W_in = np.load(f"{WEIGHTS_DIR}/W_in_canonical.npy")
        W_res = np.load(f"{WEIGHTS_DIR}/W_res_canonical.npy")
        W_out = np.load(f"{WEIGHTS_DIR}/W_out_chua_final.npy")
        return W_in, W_res, W_out
    except FileNotFoundError:
        print("CRITICAL ERROR: Weight files (.npy) not found!")
        print("Please upload W_in_canonical.npy, W_res_canonical.npy, W_out_chua_final.npy")
        raise

def activation_func(x):
    return np.tanh(x)

def skew_tent_map(x, p=0.4):
    x_norm = (x + 1.0) / 2.0
    x_norm = np.clip(x_norm, 0.001, 0.999)
    if x_norm < p: return x_norm / p
    else: return (1 - x_norm) / (1 - p)

def generate_keystream_esn(n_bytes, W_in, W_res, W_out, seed_perturbation=0.0):
    """
    Generates chaotic bytes. 
    seed_perturbation: Small change to initial condition for Sensitivity Tests.
    """
    print(f"   -> Generating {n_bytes} bytes (Perturbation={seed_perturbation})...")
    
    # Initialize Reservoir
    r = np.zeros(N_RESERVOIR)
    u = np.array([0.1 + seed_perturbation, 0.2, 0.3]) # Apply perturbation here
    
    # Warmup
    for _ in range(N_DROP):
        r = (1 - LEAK_RATE) * r + LEAK_RATE * activation_func(W_res @ r + W_in @ u)
        u = W_out @ r

    keystream = bytearray()
    generated_count = 0
    
    # Indices to tap for entropy
    tap_idx = np.linspace(0, N_RESERVOIR-1, 32, dtype=int)
    
    while generated_count < n_bytes:
        # Step ESN
        r = (1 - LEAK_RATE) * r + LEAK_RATE * activation_func(W_res @ r + W_in @ u)
        u = W_out @ r
        
        # Extract Bits (Hybrid Method)
        # 1. Reservoir Bits
        res_bits = 0
        vals = r[tap_idx]
        for v in vals:
            res_bits = (res_bits << 1) | (1 if v > 0 else 0)
            
        # 2. STM Bits
        stm_val = skew_tent_map(u[0])
        stm_bits = int(stm_val * (2**32))
        
        # 3. Mix
        final_int = res_bits ^ stm_bits
        
        # Append 4 bytes
        keystream.extend(int.to_bytes(final_int & 0xFFFFFFFF, 4, 'big'))
        generated_count += 4
        
    return np.array(keystream[:n_bytes], dtype=np.uint8)

# ==========================================
# 3. METRIC CALCULATORS
# ==========================================
def calc_entropy(img_flat):
    hist, _ = np.histogram(img_flat, bins=256, range=(0, 256))
    prob = hist / np.sum(hist)
    # Filter zeros to avoid log(0)
    prob = prob[prob > 0]
    return -np.sum(prob * np.log2(prob))

def calc_correlation(img_arr):
    # Calculate Horizontal Correlation
    x = img_arr[:, :-1].flatten()
    y = img_arr[:, 1:].flatten()
    return np.corrcoef(x, y)[0, 1]

def calc_npcr_uaci(img1_flat, img2_flat, height, width):
    # NPCR: Number of Pixels Change Rate
    diff = img1_flat != img2_flat
    npcr = (np.sum(diff) / (height * width * 3)) * 100
    
    # UACI: Unified Average Changing Intensity
    abs_diff = np.abs(img1_flat.astype(int) - img2_flat.astype(int))
    uaci = (np.sum(abs_diff) / (height * width * 3 * 255)) * 100
    
    return npcr, uaci

# ==========================================
# 4. MAIN EXECUTION
# ==========================================
# A. Setup
W_in, W_res, W_out = load_weights()
img = Image.open(IMAGE_PATH).convert('RGB')
img_arr = np.array(img)
h, w, c = img_arr.shape
total_bytes = h * w * c
print(f"Loaded {IMAGE_PATH}: {w}x{h} pixels.")

# B. Generate Keys
# Key 1: Standard
start_t = time.time()
key1 = generate_keystream_esn(total_bytes, W_in, W_res, W_out, seed_perturbation=0.0)
gen_time = time.time() - start_t

# Key 2: Perturbed (For NPCR/UACI test) -> Change seed by 10^-14
key2 = generate_keystream_esn(total_bytes, W_in, W_res, W_out, seed_perturbation=1e-14)

# C. Encrypt Images
flat_img = img_arr.flatten()

# Encrypt C1 (Key 1)
flat_c1 = np.bitwise_xor(flat_img, key1)
enc_img_arr = flat_c1.reshape(h, w, c)
Image.fromarray(enc_img_arr.astype('uint8')).save("encrypted_1.png")

# Encrypt C2 (Key 2) - For Sensitivity Test
flat_c2 = np.bitwise_xor(flat_img, key2)

# Decrypt C1 (Verify)
flat_dec = np.bitwise_xor(flat_c1, key1)
Image.fromarray(flat_dec.reshape(h, w, c).astype('uint8')).save("decrypted.png")

# ==========================================
# 5. GENERATE RESULTS TABLE
# ==========================================
print("\n" + "="*40)
print(f"RESULTS FOR IMAGE: {IMAGE_PATH}")
print("="*40)

# 1. Visual / Histogram
entropy_orig = calc_entropy(flat_img)
entropy_enc = calc_entropy(flat_c1)
print(f"1. ENTROPY ANALYSIS (Ideal = 8.0000)")
print(f"   Original:  {entropy_orig:.4f}")
print(f"   Encrypted: {entropy_enc:.4f}  <-- Should be close to 7.99")

# 2. Correlation (Horizontal)
corr_orig = calc_correlation(img_arr[:,:,0]) # Red channel
corr_enc = calc_correlation(enc_img_arr[:,:,0])
print(f"\n2. CORRELATION (Ideal = 0.0)")
print(f"   Original:  {corr_orig:.4f} (High correlation)")
print(f"   Encrypted: {corr_enc:.4f} (Should be near 0)")

# 3. Differential Attack (NPCR & UACI)
# Ideally NPCR > 99.6%, UACI ~ 33.4%
npcr, uaci = calc_npcr_uaci(flat_c1, flat_c2, h, w)
print(f"\n3. KEY SENSITIVITY (NPCR & UACI)")
print(f"   (Metric for 10^-14 change in key)")
print(f"   NPCR: {npcr:.4f}%  (Ideal > 99.60%)")
print(f"   UACI: {uaci:.4f}%  (Ideal ~ 33.46%)")

# 4. Speed
print(f"\n4. SPEED PERFORMANCE")
print(f"   Keystream Gen Time: {gen_time:.4f} sec")
print(f"   Throughput (Est):   {(total_bytes*8 / gen_time / 1e6):.2f} Mbps")

# ==========================================
# 6. PLOTTING
# ==========================================
fig, ax = plt.subplots(2, 2, figsize=(10, 8))

# Images
ax[0,0].imshow(img_arr)
ax[0,0].set_title("Original Image")
ax[0,0].axis('off')

ax[0,1].imshow(enc_img_arr)
ax[0,1].set_title("Encrypted Image")
ax[0,1].axis('off')

# Histograms (Red Channel)
ax[1,0].hist(img_arr[:,:,0].flatten(), bins=256, color='red', alpha=0.7)
ax[1,0].set_title("Original Histogram")

ax[1,1].hist(enc_img_arr[:,:,0].flatten(), bins=256, color='red', alpha=0.7)
ax[1,1].set_title("Encrypted Histogram\n(Uniform = Secure)")

plt.tight_layout()
plt.savefig("analysis_result.png")
plt.show()
print("Analysis plot saved to 'analysis_result.png'")